In [2]:
import datasets

In [ ]:
path = "SlavaYus/tinysms_qa_12m"
dataset = datasets.load_dataset(path)
# split = dataset['train'].train_test_split(test_size=0.1)

DatasetDict({
    train: Dataset({
        features: ['question', 'code'],
        num_rows: 11838003
    })
    test: Dataset({
        features: ['question', 'code'],
        num_rows: 1919
    })
})

In [9]:
questions = {row["question"] for row in dataset['train'].take(1000000)}
print(len(questions))

953764


In [12]:
for i in range(1):
    sample = dataset['test'][1]
    print('question:')
    print(sample['question'])
    print()
    print('code:')
    print(sample['code'])
    print('=' * 40)
    

question:
Student: The area of a triangle is 54 square inches. The base of the triangle is 12 inches. Find the height of the triangle in inches.

code:
None


In [11]:
sum(row['code'] is None for row in dataset['test'])

1919

In [15]:
tokenized_dataset = datasets.load_from_disk('artifacts/tokenized_dataset')

In [16]:
import yaml
with open("tokenizer.yml", "r", encoding="utf-8") as f:
    tokenizer_config =  yaml.safe_load(f)


In [19]:
from tokenizer import build_tokenizer
from collate import build_collator

tokenizer = build_tokenizer(tokenizer_config)
collator = build_collator(tokenizer_config, tokenizer)

In [26]:
print(tokenized_dataset['train'][:3])

{'input_ids': [[2589, 260, 306, 1529, 17, 518, 1247, 193, 1529, 154, 407, 2855, 916, 214, 3711, 1593, 249, 1096, 317, 595, 2361, 17, 278, 238, 1529, 362, 1702, 382, 569, 34], [36, 1425, 183, 5939, 3347, 1092, 324, 165, 4153, 17, 1334, 4866, 256, 230, 360, 214, 1829, 165, 417, 2395, 1626, 17, 279, 536, 4866, 174, 178, 256, 306, 491, 15, 682, 595, 906, 609, 586, 174, 491, 18, 709, 34], [36, 676, 244, 447, 301, 172, 214, 1426, 822, 5360, 171, 4030, 244, 1388, 611, 171, 3330, 162, 11, 87, 12, 146, 185, 3470, 87, 65, 21, 221, 1966, 87, 221, 345, 17, 592, 244, 171, 2537, 822, 183, 171, 676, 5360, 171, 4030, 34]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [

In [31]:
from torch.utils.data import DataLoader

dataloader = DataLoader(tokenized_dataset['train'], collate_fn=collator, batch_size=4)

In [40]:
print((next(iter(dataloader))))

{'input_ids': tensor([[2589,  260,  306, 1529,   17,  518, 1247,  193, 1529,  154,  407, 2855,
          916,  214, 3711, 1593,  249, 1096,  317,  595, 2361,   17,  278,  238,
         1529,  362, 1702,  382,  569,   34,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0],
        [  36, 1425,  183, 5939, 3347, 1092,  324,  165, 4153,   17, 1334, 4866,
          256,  230,  360,  214, 1829,  165,  417, 2395, 1626,   17,  279,  536,
         4866,  174,  178,  256,  306,  491,   15,  682,  595,  906,  609,  586,
          174,  491,   18,  709,   34,    0,    0,    0,    0],
        [  36,  676,  244,  447,  301,  172,  214, 1426,  822, 5360,  171, 4030,
          244, 1388,  611,  171, 3330,  162,   11,   87,   12,  146,  185, 3470,
           87,   65,   21,  221, 1966,   87,  221,  345,   17,  592,  244,  171,
         2537,  822,  183,  171,  676, 5360,  171, 4030,   34],
        [7881,  260,  249,  525,  171,  322,  183, 1529,  463,  3

In [18]:
tokenizer_train_dataset = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(2_000_000))
)

In [18]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


tokenizer = Tokenizer(
    BPE(
        unk_token="<unk>"
    )
)

tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=[
        "<pad>",
        "<unk>",
        "<bos>",
        "<eos>",
    ],
)

In [19]:
def batch_iterator(dataset, batch_size=10_000):
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i:i + batch_size]

        texts = []

        for question, code in zip(
            batch["question"],
            batch["code"]
        ):
            texts.append(question)
            texts.append(code)

        yield texts

In [20]:

tokenizer.train_from_iterator(
    batch_iterator(tokenizer_train_dataset),
    trainer=trainer,
    length=len(tokenizer_train_dataset) * 2,
)

In [21]:
tokenizer.save("tokenizer.json")